# 🏛️ VietLawAssist — Fault-Tolerant Modular QLoRA Training Pipeline
### Đồ án Chuyên ngành PBL6 — Đại học Bách Khoa Đà Nẵng (DUT)

**Kiến trúc Chống Đứt Gãy (Fault-Tolerant & State Persistence)**:
- **Block 1**: Thiết lập môi trường & Kiểm tra GPU (Có cơ chế kiểm tra import tránh cài lại).
- **Block 2**: Tải & Lưu Base Model Snapshot cục bộ (Tự động Resume Download, khi đã tải xong thì chuyển 100% Offline).
- **Block 3**: Tokenize & Cache Dataset dạng Arrow trên đĩa (Nếu crash nạp lại chỉ mất 0.1 giây).
- **Block 4**: Huấn luyện QLoRA với Auto-Resume Checkpoints (Nếu rớt mạng/GPU preemption sẽ chạy tiếp từ step gần nhất).
- **Block 5**: Xuất bản & Nén file Zip Adapter nhẹ (~25MB - 100MB).
- **Block 6**: Kiểm thử suy luận 100% Offline (Không tốn 1 byte internet).
- **Block 7**: Trực quan hóa biểu đồ Loss thời gian thực.

In [ ]:
# =============================================================================
# BLOCK 1: THIẾT LẬP THƯ VIỆN & KIỂM TRA PHẦN CỨNG
# =============================================================================
import sys

def ensure_dependencies():
    try:
        import transformers, datasets, peft, trl, bitsandbytes, accelerate, loguru
        print("[*] Tất cả các thư viện cần thiết đã sẵn sàng. BỎ QUA cài đặt lại!")
    except ImportError:
        print("[*] Đang cài đặt các thư viện mới nhất...")
        !pip install -q --upgrade transformers datasets peft trl bitsandbytes accelerate loguru

ensure_dependencies()

import os
import torch
from pathlib import Path

print("----------------------------------------------------------------------")
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] GPU khả dụng: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CHƯA BẬT GPU (Hãy chọn GPU T4 x2 hoặc P100)'}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[*] VRAM: {vram:.2f} GB")
print("----------------------------------------------------------------------")

In [ ]:
# =============================================================================
# BLOCK 2: TẢI VÀ LƯU BASE MODEL CỤC BỘ (LOCAL SNAPSHOT RESUME)
# Nếu đã có file trên đĩa -> BỎ QUA TẢI, kích hoạt chế độ 100% OFFLINE!
# =============================================================================
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Có thể đổi thành Qwen/Qwen2.5-Coder-7B-Instruct
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/input").exists() else Path("./kaggle/output")
LOCAL_MODEL_DIR = WORKING_DIR / "local_base_model"
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

config_file = LOCAL_MODEL_DIR / "config.json"
has_weights = any(LOCAL_MODEL_DIR.glob("*.safetensors")) or any(LOCAL_MODEL_DIR.glob("*.bin"))

if config_file.exists() and has_weights:
    print(f"[SUCCESS] Base Model ĐÃ CÓ SẴN trên ổ cứng cục bộ tại:\n  -> {LOCAL_MODEL_DIR}")
    print("[*] Kích hoạt chế độ 100% OFFLINE! Không cần kết nối Internet.")
else:
    print(f"[*] Đang tải Base Model '{MODEL_ID}' về: {LOCAL_MODEL_DIR}...")
    print("[*] Kích hoạt tính năng 'resume_download=True' (Nếu đứt mạng, chạy lại cell này sẽ tải tiếp!)")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(LOCAL_MODEL_DIR),
        local_dir_use_symlinks=False,
        resume_download=True,
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"]
    )
    print(f"[SUCCESS] Đã tải trọn vẹn Base Model vào thư mục cục bộ!")

In [ ]:
# =============================================================================
# BLOCK 3: TIỀN XỬ LÝ & LƯU TOKENIZED DATASET DẠNG ARROW (DISK CACHE)
# Nếu đã có cache -> Nạp tức thì trong 0.1 giây mà không cần format lại
# =============================================================================
import json
from datasets import Dataset, load_from_disk
from transformers import AutoTokenizer

CACHED_DATA_DIR = WORKING_DIR / "cache_tokenized_dataset"
train_cache = CACHED_DATA_DIR / "train"
val_cache = CACHED_DATA_DIR / "val"

if train_cache.exists() and val_cache.exists():
    print(f"[SUCCESS] Đã tìm thấy Dataset Cache trên đĩa tại:\n  -> {CACHED_DATA_DIR}")
    train_dataset = load_from_disk(str(train_cache))
    val_dataset = load_from_disk(str(val_cache))
    print(f"[*] Nạp siêu tốc: Train={len(train_dataset)} mẫu | Val={len(val_dataset)} mẫu (0.1s)")
else:
    print("[*] Đang đọc tệp JSON và format dữ liệu theo chuẩn ChatML...")
    # Tìm thư mục input chứa dataset
    dataset_input_dir = None
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").glob("*"):
            if (p / "train_sft_chatml.json").exists():
                dataset_input_dir = p
                break
        if not dataset_input_dir:
            dataset_input_dir = Path("/kaggle/input/vietlawassist-dataset")
    else:
        dataset_input_dir = Path("./kaggle/dataset")

    train_json = dataset_input_dir / "train_sft_chatml.json"
    val_json = dataset_input_dir / "val_sft_chatml.json"

    with open(train_json, "r", encoding="utf-8") as f: train_raw = json.load(f)
    with open(val_json, "r", encoding="utf-8") as f: val_raw = json.load(f)

    tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_MODEL_DIR), local_files_only=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    def format_list(samples):
        texts = []
        for s in samples:
            if "messages" in s:
                t = tokenizer.apply_chat_template(s["messages"], tokenize=False, add_generation_prompt=False)
            else:
                msgs = [
                    {"role": "system", "content": "Bạn là VietLawAssist — Trợ lý AI Cố vấn Học thuật Pháp luật Đại cương tại ĐHBK Đà Nẵng (DUT)."},
                    {"role": "user", "content": f"{s.get('instruction', '')}\n\n{s.get('input', '')}".strip()},
                    {"role": "assistant", "content": s.get("output", "").strip()}
                ]
                t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            texts.append(t)
        return Dataset.from_dict({"text": texts})

    train_dataset = format_list(train_raw)
    val_dataset = format_list(val_raw)
    
    # Lưu ra đĩa
    train_dataset.save_to_disk(str(train_cache))
    val_dataset.save_to_disk(str(val_cache))
    print(f"[SUCCESS] Đã tiền xử lý và lưu Cache thành công:\n  - Train: {len(train_dataset)} mẫu\n  - Val: {len(val_dataset)} mẫu")

In [ ]:
# =============================================================================
# BLOCK 4: HUẤN LUYỆN QLORA VỚI CƠ CHẾ AUTO-RESUME TỪ CHECKPOINT CŨ
# Nếu đang train bị rớt mạng / crash -> Chạy lại cell này sẽ TIẾP TỤC từ step gần nhất!
# =============================================================================
import time, csv
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_CSV = WORKING_DIR / "training_metrics.csv"

# 1. Nạp Tokenizer & Model dạng 4-bit NF4 từ Local Model (100% Offline)
tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_MODEL_DIR), local_files_only=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

print("[*] Nạp Base Model 4-bit từ thư mục cục bộ...")
model = AutoModelForCausalLM.from_pretrained(
    str(LOCAL_MODEL_DIR),
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True
)
model = prepare_model_for_kbit_training(model)

# 2. Cấu hình LoRA
peft_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 3. Data Collator Prompt Loss Masking
collator = DataCollatorForCompletionOnlyLM(
    response_template="<|im_start|>assistant\n",
    tokenizer=tokenizer
)

# 4. Training Arguments
training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    dataset_text_field="text",
    max_seq_length=1536,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    optim="paged_adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    evaluation_strategy="steps",
    eval_steps=25,
    logging_steps=5,
    seed=42
)

# Real-time CSV logging callback
class CSVLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        with open(METRICS_CSV, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([state.global_step, round(state.epoch or 0, 2), logs.get("loss", ""), logs.get("eval_loss", "")])

if not METRICS_CSV.exists():
    with open(METRICS_CSV, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["step", "epoch", "loss", "eval_loss"])

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    callbacks=[CSVLogCallback()]
)

# 5. TỰ ĐỘNG PHÁT HIỆN CHECKPOINT CŨ ĐỂ RESUME
existing_ckpts = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
last_checkpoint = str(existing_ckpts[-1]) if existing_ckpts else None

if last_checkpoint:
    print(f"\n[!] ==============================================================")
    print(f"[!] PHÁT HIỆN CHECKPOINT CŨ TẠI: {last_checkpoint}")
    print(f"[!] TỰ ĐỘNG TIẾP TỤC HUẤN LUYỆN TỪ CHECKPOINT NÀY! (Không mất bước cũ)")
    print(f"[!] ==============================================================\n")
else:
    print("\n[*] Bắt đầu huấn luyện từ Step 0 (Epoch 1/3)...\n")

start_time = time.time()
trainer.train(resume_from_checkpoint=last_checkpoint)
print(f"\n[SUCCESS] Huấn luyện hoàn thành trong {(time.time() - start_time)/60:.2f} phút!")

In [ ]:
# =============================================================================
# BLOCK 5: ĐÓNG GÓI & XUẤT BẢN FINAL LORA ADAPTER
# =============================================================================
import shutil

FINAL_ADAPTER_DIR = WORKING_DIR / "vietlaw_lora_adapter"
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"[*] Đang lưu trọng số LoRA và Tokenizer vào: {FINAL_ADAPTER_DIR}...")
model.save_pretrained(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

zip_path = WORKING_DIR / "vietlaw_lora_adapter"
shutil.make_archive(str(zip_path), "zip", str(FINAL_ADAPTER_DIR))
zip_size_mb = Path(f"{zip_path}.zip").stat().st_size / (1024**2)

print(f"[SUCCESS] Đã đóng gói thành công file: {zip_path}.zip ({zip_size_mb:.2f} MB)")
print("[*] Bạn có thể bấm vào mục Output bên cột phải của Kaggle để tải file zip về máy!")

In [ ]:
# =============================================================================
# BLOCK 6: BÀI TEST SUY LUẬN OFFLINE (ZERO-INTERNET INFERENCE TEST)
# =============================================================================
from peft import PeftModel

test_queries = [
    "Tình huống: Ông A chết có tài sản chung với vợ là 1,2 tỷ đồng, có 2 con đẻ C (15 tuổi) và D (22 tuổi đã đi làm). Ông A lập di chúc cho bạn thân Q toàn bộ tài sản. Hãy xác định ai được hưởng di sản theo Điều 644 BLDS 2015 và tính giá trị mỗi suất?",
    "Tình huống: Anh Hải (20 tuổi) lái xe máy chở bạn vượt đèn đỏ với tốc độ 70km/h trong khu dân cư và đâm vào người đi bộ gây chấn thương 35%. Hãy xác định dạng lỗi của Hải và giải thích."
]

model.eval()
for i, q in enumerate(test_queries, 1):
    print(f"==================================================================")
    print(f"TEST CASE {i}:\n{q}")
    print(f"------------------------------------------------------------------")
    msgs = [
        {"role": "system", "content": "Bạn là VietLawAssist — Trợ lý AI Cố vấn Học thuật Pháp luật Đại cương tại ĐHBK Đà Nẵng (DUT)."},
        {"role": "user", "content": q}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.2, top_p=0.9)
    res = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"[VIETLAWASSIST LỜI GIẢI]:\n{res}\n")

In [ ]:
# =============================================================================
# BLOCK 7: TRỰC QUAN HÓA ĐƯỜNG CONG LOSS (TRAINING & VALIDATION LOSS)
# =============================================================================
import pandas as pd
import matplotlib.pyplot as plt

if METRICS_CSV.exists():
    df = pd.read_csv(METRICS_CSV)
    df["loss"] = pd.to_numeric(df["loss"], errors="coerce")
    df["eval_loss"] = pd.to_numeric(df["eval_loss"], errors="coerce")
    
    plt.figure(figsize=(10, 5))
    plt.plot(df["step"], df["loss"].interpolate(), label="Train Loss", color="blue", lw=2)
    eval_df = df.dropna(subset=["eval_loss"])
    if not eval_df.empty:
        plt.plot(eval_df["step"], eval_df["eval_loss"], label="Validation Loss", color="red", marker="o", lw=2)
        
    plt.title("VietLawAssist — QLoRA Training & Validation Loss Curve")
    plt.xlabel("Global Steps")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.savefig(str(WORKING_DIR / "loss_curve.png"), dpi=300)
    plt.show()
    print("[*] Biểu đồ loss đã được lưu tại: loss_curve.png")
else:
    print("[*] Chưa tìm thấy training_metrics.csv để vẽ biểu đồ.")